In [2]:
!pip install torch pandas scikit-learn numpy matplotlib


[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: C:\Users\chara\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [3]:
print("All required packages have been installed successfully.")

All required packages have been installed successfully.


In [4]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
from torch.utils.data import TensorDataset, DataLoader

# 1. Load both training subsets
df1 = pd.read_csv("C:\\5g usecase\\datasets\\Train_subset_1.csv")
df2 = pd.read_csv("C:\\5g usecase\\datasets\\Train_subset_2.csv")

# 2. Combine them into one massive dataset
df = pd.concat([df1, df2], ignore_index=True)
print(f"Combined dataset shape: {df.shape}")

# Find the exact name of the label column
label_col = df.columns[-1] 
print(f"Assuming '{label_col}' is the target label column.")

# 3. Filter for NORMAL traffic only (assuming 0 is Normal)
normal_df = df[df[label_col] == 0].copy()
print(f"Normal traffic shape (Attacks dropped): {normal_df.shape}")

# 4. Drop the label column entirely (Unsupervised Learning)
X_train = normal_df.drop(columns=[label_col])

# 5. Scale the features to a standard range (Mean=0, Variance=1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# 6. Convert to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)

# 7. Create a DataLoader to feed data in batches of 64
train_dataset = TensorDataset(X_train_tensor, X_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print(f"\nSuccess, bro! First batch shape: {next(iter(train_loader))[0].shape}")

Combined dataset shape: (30288, 45)
Assuming 'label' is the target label column.
Normal traffic shape (Attacks dropped): (23888, 45)

Success, bro! First batch shape: torch.Size([64, 44])


In [5]:
import torch.nn as nn
import torch.optim as optim

# 1. Define the Transformer Autoencoder Architecture
class TransformerAutoencoder(nn.Module):
    def __init__(self, input_dim=44, d_model=64, nhead=4, num_layers=2, latent_dim=16):
        super(TransformerAutoencoder, self).__init__()
        
        # Encoder Projection: Scales 44 features up to 64 for the Transformer
        self.input_proj = nn.Linear(input_dim, d_model)
        
        # Transformer Encoder: Learns complex relationships between network features
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # The Bottleneck: Compresses data down to a tiny 16-dimension latent space
        self.to_latent = nn.Linear(d_model, latent_dim)
        
        # Decoder Projection: Expands back to 64
        self.from_latent = nn.Linear(latent_dim, d_model)
        
        # Transformer Decoder: Tries to reconstruct the patterns
        decoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_decoder = nn.TransformerEncoder(decoder_layer, num_layers=num_layers)
        
        # Output Projection: Scales back down to exactly 44 features
        self.output_proj = nn.Linear(d_model, input_dim)
        
    def forward(self, x):
        # x starts as (Batch, 44). We add a sequence dimension -> (Batch, 1, 44)
        x = x.unsqueeze(1)
        
        # Encode
        x = self.input_proj(x)
        x = self.transformer_encoder(x)
        latent = self.to_latent(x)
        
        # Decode
        x = self.from_latent(latent)
        x = self.transformer_decoder(x)
        out = self.output_proj(x)
        
        # Remove sequence dimension -> (Batch, 44)
        return out.squeeze(1)

# 2. Initialize the Model, Loss Function, and Optimizer
model = TransformerAutoencoder(input_dim=44)
criterion = nn.MSELoss() # Mean Squared Error calculates the Reconstruction Error
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. Train the Model (Unsupervised Learning on Normal Traffic)
epochs = 10
print("Starting training, bro. Watch the loss drop...\n")

for epoch in range(epochs):
    total_loss = 0
    for batch in train_loader:
        # Get the input data (inputs and targets are the same for autoencoders)
        inputs = batch[0]
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, inputs)
        
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] | Reconstruction Loss: {avg_loss:.6f}")

print("\nModel trained successfully! It now knows exactly what a healthy 5G network looks like.")

Starting training, bro. Watch the loss drop...

Epoch [1/10] | Reconstruction Loss: 0.129620
Epoch [2/10] | Reconstruction Loss: 0.064770
Epoch [3/10] | Reconstruction Loss: 0.062579
Epoch [4/10] | Reconstruction Loss: 0.089792
Epoch [5/10] | Reconstruction Loss: 0.058322
Epoch [6/10] | Reconstruction Loss: 0.049151
Epoch [7/10] | Reconstruction Loss: 0.059258
Epoch [8/10] | Reconstruction Loss: 0.079320
Epoch [9/10] | Reconstruction Loss: 0.049297
Epoch [10/10] | Reconstruction Loss: 0.050105

Model trained successfully! It now knows exactly what a healthy 5G network looks like.


In [6]:
import torch
import joblib
import os

# Create a folder to keep your saved models organized
os.makedirs("models", exist_ok=True)

# 1. Save the PyTorch Transformer Autoencoder weights
model_path = "models/transformer_autoencoder.pth"
torch.save(model.state_dict(), model_path)
print(f"✅ PyTorch model securely saved at: {model_path}")

# 2. Save the Scaler (CRITICAL for the FastAPI server)
# Assuming your scaler variable is named 'scaler' from your preprocessing steps
scaler_path = "models/scaler.pkl"
joblib.dump(scaler, scaler_path)
print(f"✅ Data Scaler securely saved at: {scaler_path}")

✅ PyTorch model securely saved at: models/transformer_autoencoder.pth
✅ Data Scaler securely saved at: models/scaler.pkl


In [8]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# 1. Load the Test Data (The mixed dataset with attacks)
test_df = pd.read_csv("C:\\5g usecase\\datasets\\Test_Data.csv")
print(f"Test dataset loaded! Shape: {test_df.shape}")

# 2. Separate features and true labels
label_col = test_df.columns[-1]
y_true = test_df[label_col].values  # 0 for Normal, 1 for Attack
X_test = test_df.drop(columns=[label_col])

# 3. Scale using the EXACT SAME scaler we fitted during training
X_test_scaled = scaler.transform(X_test)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

# 4. Put the model in evaluation mode (turns off dropout/gradients)
model.eval()

print("\nRunning test data through the Transformer Autoencoder...")
# 5. Calculate Reconstruction Error for every single packet
with torch.no_grad(): 
    reconstructed = model(X_test_tensor)
    
    # Calculate Mean Squared Error (MSE) for each row individually
    # Instead of average batch loss, we want the error score for every single packet
    mse_per_row = torch.mean((X_test_tensor - reconstructed) ** 2, dim=1).numpy()

# 6. Set the Anomaly Threshold
# Since your training loss ended at ~0.04, any packet with an error > 0.08 is highly suspicious
THRESHOLD = 0.08 

# 7. Make Predictions: If error > threshold, flag as Attack (1). Else, Normal (0).
y_pred = (mse_per_row > THRESHOLD).astype(int)

# 8. Print the Cyber Security Evaluation Metrics!
print("\n--- 5G Anomaly Detection Results ---")
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))
print("\nClassification Report:")
print(classification_report(y_true, y_pred))

Test dataset loaded! Shape: (3787, 45)

Running test data through the Transformer Autoencoder...

--- 5G Anomaly Detection Results ---
Confusion Matrix:
[[2946   41]
 [   0  800]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      2987
           1       0.95      1.00      0.98       800

    accuracy                           0.99      3787
   macro avg       0.98      0.99      0.98      3787
weighted avg       0.99      0.99      0.99      3787



In [9]:
pip install langchain-groq langgraph langchain-core pydantic

Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
import numpy as np
import pandas as pd
from typing import TypedDict
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

# 1. SET YOUR NEW API KEY HERE
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "your_groq_api_key_here")

# 2. Define the Graph State
class SOCState(TypedDict):
    packet_data: str
    threat_report: str

# 3. Extract the first malicious packet
malicious_indices = np.where(y_pred == 1)[0]
first_attack_idx = malicious_indices[0]
flagged_packet = test_df.iloc[first_attack_idx].to_dict()
print(f"Intercepted Malicious Packet at Index {first_attack_idx}!\nRouting to LLM Analyst...\n")

# 4. Define the Agentic Node
def analyze_threat(state: SOCState):
    # UPDATED: Using OpenAI's flagship open-weight 120B model on Groq
    llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.2)
    
    prompt = f"""
    You are an elite 5G Network Cybersecurity Analyst daemon. 
    Our PyTorch Transformer Autoencoder just flagged the following network packet as a critical anomaly.
    
    Analyze this raw 5G packet data:
    {state['packet_data']}
    
    Generate a concise, professional Threat Incident Report. Include exactly these three sections:
    1. Threat Diagnosis (What kind of attack does this look like based on the ports/flags/lengths?)
    2. Severity Level (Critical, High, Medium, or Low)
    3. Autonomous Action (Suggest a specific firewall rule or IP block to mitigate this instantly).
    """
    
    response = llm.invoke(prompt)
    return {"threat_report": response.content}

# 5. Compile and Execute the LangGraph Workflow
workflow = StateGraph(SOCState)
workflow.add_node("analyst_agent", analyze_threat)
workflow.set_entry_point("analyst_agent")
workflow.add_edge("analyst_agent", END)
soc_app = workflow.compile()

initial_state = {"packet_data": str(flagged_packet)}
result = soc_app.invoke(initial_state)

print("--- AUTONOMOUS 5G THREAT REPORT ---")
print(result['threat_report'])

Intercepted Malicious Packet at Index 1!
Routing to LLM Analyst...

--- AUTONOMOUS 5G THREAT REPORT ---
**Threat Diagnosis**  
The packet is a minimal‑size IPv4 frame (28 bytes) with **IP proto = 1 (ICMP)** and **no TCP/UDP ports**. All TCP/UDP fields are zero, and GTP fields contain invalid sentinel values (‑1). This pattern matches a **malformed or spoofed ICMP Echo‑Request used for reconnaissance/flooding** (often seen in ICMP‑based scanning or low‑rate DoS attempts). The presence of impossible GTP flag/TEID values further indicates a crafted packet intended to confuse protocol parsers.

**Severity Level**  
**High** – The anomaly deviates sharply from normal traffic baselines, suggests active probing or a potential ICMP‑flood precursor, and could be a stepping stone for larger denial‑of‑service or tunneling attacks on the 5G core.

**Autonomous Action**  
Deploy an immediate firewall rule that drops any inbound ICMP Echo‑Request (type 8) with an IP payload length of 8 bytes (total 